# GNN — Physical Experience Comparison

Trains the EGNN (v8 settings) on one of the 6 physical experiences, **with** and **without** physics in the loss,
then shows a side-by-side Stage 0 / Stage 1 / Stage 2 comparison.

**Change `EXP_NUM` in the cell below to select the experience (1–6).**

| Exp | Clamp | Load description |
|-----|-------|------------------|
| 1 | face 0 | face 14, dof 2 (rot), −4 |
| 2 | none | 4-point lateral: faces [1,4]↑ [10,15]↓ [3,8]← [13,6]→ |
| 3 | face 8 | counter-rotate: face 12 +10, face 7 −10 |
| 4 | none | double counter-rotate: [12,2]+10, [7,9]−10 |
| 5 | face 1 | single large moment: face 4, dof 2, +15 |
| 6 | face 1 | moment far from clamp: face 10, dof 2, +10 |

In [ ]:
import os, sys
os.environ["JAX_PLATFORM_NAME"] = "cpu"

_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, _root)
sys.path.insert(0, os.path.join(_root, 'src'))

import jax
jax.config.update("jax_enable_x64", True)

import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import copy
from types import SimpleNamespace

from topology.builder import build_tessellation
from problem.conditions import configure_tessellation
from problem.config import load_and_parse_config
from jax_backend.state import CentroidalState
from jax_backend.pipeline import forward_pipeline
from jax_backend.training.trainer import train_pipeline
from jax_backend.gnn.graph_builder import build_static_graph_features
from jax_backend.gnn.egnn import init_egnn
from jax_backend.geometry import reconstruct_vertices, compute_face_areas
from jax_backend.utils.linalg import rotation_matrix
from utils.visualization import plot_tessellation

print(f"JAX backend: {jax.default_backend()}")

In [ ]:
# ── SELECT EXPERIMENT ─────────────────────────────────────────────────────────
EXP_NUM = 1   # Change to 1–6

CONFIG_DIR = os.path.join(_root, "data/configs/gnn")
config_phys_path   = os.path.join(CONFIG_DIR, f"exp{EXP_NUM}_gnn.yaml")
config_nophys_path = os.path.join(CONFIG_DIR, f"exp{EXP_NUM}_gnn_nophys.yaml")

print(f"Experience {EXP_NUM}")
print(f"  With physics:    {config_phys_path}")
print(f"  Without physics: {config_nophys_path}")

In [ ]:
# ── HELPERS ───────────────────────────────────────────────────────────────────

def setup_experiment(config_path):
    """Load config, build tessellation, center for GNN, init EGNN params."""
    config = load_and_parse_config(config_path)
    topo = config.topology
    topo_obj = SimpleNamespace(**topo)

    tessellation = build_tessellation(topo.get('pattern'), topo.get('width', 5), topo.get('height', 5))

    requested_area = topo.get('total_area')
    if requested_area:
        scale = np.sqrt(requested_area / tessellation.compute_total_area())
        tessellation.update_vertices(tessellation.vertices * scale)

    configure_tessellation(tessellation, topo_obj)

    # Center tessellation on target (required for GNN positional encoding)
    target_center = np.array(getattr(config.target, 'center', [0.0, 0.0]), dtype=float)
    tess_centroid = np.mean(tessellation.get_face_centroids(), axis=0)
    tessellation.update_vertices(tessellation.vertices - tess_centroid + target_center)

    initial_state = CentroidalState.from_tessellation(tessellation, target_cfg=config.target)

    # Init EGNN
    static_features = build_static_graph_features(initial_state)
    node_feat_dim = static_features['node_feat_dim']
    gnn_cfg = config.mapping.params if isinstance(config.mapping.params, dict) else {}
    hidden_dim = int(gnn_cfg.get('hidden_dim', 64))
    num_layers = int(gnn_cfg.get('num_layers', 4))
    seed = int(gnn_cfg.get('seed', 2))
    key = jax.random.PRNGKey(seed)
    init_params = init_egnn(key, node_feat_dim, hidden_dim, num_layers)

    return config, tessellation, initial_state, init_params


def run_training(config, tessellation, initial_state, init_params):
    """Train EGNN and run final forward pass. Returns (result, history, opt_params)."""
    optimized_params, history = train_pipeline(
        init_params, initial_state,
        config.target, config.validity, config.physics, config.training,
        map_type=config.mapping.type,
        use_shirley_chiu=config.mapping.use_shirley_chiu,
        strict_boundary_fit=config.mapping.strict_boundary_fit,
        learn_global_scale=config.mapping.learn_global_scale,
        use_jit=False,  # JIT disabled for GNN on Metal/Mac
    )

    result = forward_pipeline(
        initial_state, config.target, config.validity, config.physics,
        map_type=config.mapping.type,
        map_params=optimized_params,
        use_shirley_chiu=config.mapping.use_shirley_chiu,
        strict_boundary_fit=config.mapping.strict_boundary_fit,
    )
    return result, history, optimized_params


def tess_from_state(state, tessellation):
    """Reconstruct tessellation vertices from a CentroidalState."""
    verts_rec = np.array(reconstruct_vertices(state.face_centroids, state.centroid_node_vectors))
    tess_copy = copy.deepcopy(tessellation)
    new_verts = np.zeros_like(tess_copy.vertices)
    for i, face in enumerate(tess_copy.faces):
        for j, v_idx in enumerate(face.vertex_indices):
            new_verts[v_idx] = verts_rec[i, j]
    tess_copy.update_vertices(new_verts)
    return tess_copy


def equilibrium_state(valid_state, solution):
    """Compute the Stage 2 equilibrium CentroidalState from physics solution."""
    final_fields = solution.fields[-1]            # (n_faces, 3)
    c_eq = valid_state.face_centroids + final_fields[:, :2]
    R = jax.vmap(rotation_matrix)(final_fields[:, 2])  # (n_faces, 2, 2)
    # Rotate each CNV: s_eq[n,k,i] = sum_j R[n,i,j] * cnv[n,k,j]
    s_eq = jnp.einsum('nij, nkj -> nki', R, valid_state.centroid_node_vectors)
    return valid_state._replace(face_centroids=c_eq, centroid_node_vectors=s_eq)


print("Helpers ready.")

## Train — WITH physics

GNN receives gradient from chamfer + hinge_gap + stretching + shearing.

In [ ]:
print(f"Setting up Exp {EXP_NUM} WITH physics...")
cfg_p, tess_p, s0_p, params0_p = setup_experiment(config_phys_path)
print(f"Training ({cfg_p.training.num_epochs} epochs, no JIT)...")
res_p, hist_p, opt_p = run_training(cfg_p, tess_p, s0_p, params0_p)
print("Done.")

## Train — WITHOUT physics

GNN receives gradient only from chamfer + hinge_gap. Physics solver still runs in Stage 2.

In [ ]:
print(f"Setting up Exp {EXP_NUM} WITHOUT physics...")
cfg_n, tess_n, s0_n, params0_n = setup_experiment(config_nophys_path)
print(f"Training ({cfg_n.training.num_epochs} epochs, no JIT)...")
res_n, hist_n, opt_n = run_training(cfg_n, tess_n, s0_n, params0_n)
print("Done.")

## Loss curves

In [ ]:
def extract_series(history, key):
    return [float(h[key]) for h in history if key in h]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, history, label in zip(axes, [hist_p, hist_n], ["With physics", "No physics"]):
    total   = extract_series(history, 'total')
    chamfer = extract_series(history, 'chamfer')
    hinge   = extract_series(history, 'hinge_gap')
    ax.plot(total,   label='Total',      lw=1.5)
    ax.plot(chamfer, label='Chamfer',    lw=1.2, linestyle='--')
    ax.plot(hinge,   label='Hinge gap',  lw=1.2, linestyle=':')
    ax.set_title(f"Exp {EXP_NUM} — {label}")
    ax.set_xlabel("Epoch")
    ax.legend()
    ax.set_yscale('log')
plt.tight_layout()
plt.show()

## Side-by-side: Stage 0 | Stage 1 | Stage 2

**Top row**: WITH physics — **Bottom row**: WITHOUT physics

In [ ]:
target_params = {
    'type':   cfg_p.target.type,
    'center': cfg_p.target.center,
    'radius': cfg_p.target.radius,
}
plot_kw = dict(
    show_target=True,
    target_params=target_params,
    show_hinges=True,
    show_hinge_indices=False,
    show_face_indices=False,
    show_external_forces=True,
    show_kinematic_blocks=True,
)

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle(
    f"Experience {EXP_NUM}  |  Top: WITH physics  |  Bottom: NO physics",
    fontsize=14, y=1.01
)

col_titles = ["Stage 0: Initial Mapping", "Stage 1: Geometric Validity", "Stage 2: Physical Equilibrium"]
for col, title in enumerate(col_titles):
    axes[0, col].set_title(title, fontsize=11)

for row, (res, tess, cfg, label) in enumerate([
    (res_p, tess_p, cfg_p, "WITH physics"),
    (res_n, tess_n, cfg_n, "NO  physics"),
]):
    axes[row, 0].set_ylabel(label, fontsize=10)

    # Stage 0
    t0 = tess_from_state(res['mapped_state'], tess)
    plot_tessellation(t0, ax=axes[row, 0], title="", original_vertices=tess.vertices, **plot_kw)

    # Stage 1
    t1 = tess_from_state(res['valid_state'], tess)
    plot_tessellation(t1, ax=axes[row, 1], title="", original_vertices=tess.vertices, **plot_kw)

    # Stage 2: apply physics displacement
    eq_state = equilibrium_state(res['valid_state'], res['solution'])
    t2 = tess_from_state(eq_state, tess)
    plot_tessellation(t2, ax=axes[row, 2], title="", original_vertices=tess.vertices,
                      show_target=True, target_params=target_params,
                      show_hinges=True, show_hinge_indices=False,
                      show_face_indices=False, show_external_forces=False,
                      show_kinematic_blocks=True)

plt.tight_layout()
plt.show()

## Quantitative summary

In [ ]:
def summarize(res, history, label):
    final = history[-1]
    total   = float(final.get('total',     float('nan')))
    chamfer = float(final.get('chamfer',   float('nan')))
    hinge   = float(final.get('hinge_gap', float('nan')))

    area0 = compute_face_areas(res['mapped_state'].centroid_node_vectors)
    area1 = compute_face_areas(res['valid_state'].centroid_node_vectors)
    delta_area = float(jnp.mean((area1 - area0) / area0) * 100)

    sol = res['solution']
    phys_energy = float(sol.energies[-1]) if sol.energies is not None else float('nan')

    print(f"  {label}")
    print(f"    Total loss:         {total:.3f}")
    print(f"    Chamfer (weighted): {chamfer:.4f}")
    print(f"    Hinge gap (wtd):    {hinge:.4f}")
    print(f"    ΔArea Stage 1:      {delta_area:+.2f}%")
    print(f"    Physics energy S2:  {phys_energy:.4f}")

print(f"Experience {EXP_NUM} — final metrics:")
print()
summarize(res_p, hist_p, "WITH physics")
print()
summarize(res_n, hist_n, "NO  physics")

---
## Try another experience
Change `EXP_NUM` in the second cell above and run **Kernel → Restart & Run All**.